# 08 · The comparison, and the decision

**The decision this notebook supports:** would I actually deploy this?

Everything so far produced a number. This notebook puts them side by side on the **held-out** set, which you have not looked at, and then asks you to write a recommendation.

One rule, and it is the whole point of having kept a held-out split: **you get one pass.** If you read these failures and then change something, this set has become development data and you need fresh questions for the next honest measurement.

In [ ]:
# Connect this notebook to the lab's saved paths, data, scorer, and configuration.
# sys controls Python's import search path; json reads/writes JSON; warnings controls warning messages.
# Putting the project root on sys.path lets `from src import lab` find our own lab module.
# The relative '..' path assumes a suitable working directory; importing lab does not load an LLM.
# CFG is a nested dictionary: CFG['model']['name'] looks inside 'model', then reads 'name'.
# filterwarnings('ignore') hides warnings, not exceptions; useful loader advice may therefore be hidden.

import sys, json, warnings
sys.path.insert(0, "..")          # project root, so `from src import lab` works
warnings.filterwarnings("ignore") # hide library deprecation noise
from src import lab
CFG = lab.cfg()
print("lab ready |", CFG["model"]["name"])


## Development results so far

Everything below came from the development split, which you were free to inspect.

In [ ]:
# Read previous run summaries to choose a candidate before the final evaluation.
# pandas turns dictionaries into a table; json is available for later saved reports.
# The dictionary comprehension includes only named runs already present on disk; .T makes them rows.
# style.format changes the displayed decimal precision, not the stored metric values.

import pandas as pd, json
runs = lab.load_runs()
# Each pair maps a saved run name to its readable label. The order controls presentation,
# not model loading. Prompt/context/runtime differences can confound historical comparisons.
order = [("01_stock_naive","1 stock, naive prompt"), ("02_stock_engineered","2 stock, engineered prompt"),
         ("03_stock_with_docs","3 stock + documents"), ("08_sft_eval","4 after SFT"),
         ("10_dpo_eval","5 after DPO"), ("11_quantised_eval","6 quantised 4-bit")]
rows = {label: runs[k]["summary"] for k, label in order if k in runs}
dev_df = pd.DataFrame(rows).T
cols = ["accuracy_answerable","hallucination_rate_answerable","over_refusal_rate","appropriate_refusal_rate"]
display(dev_df[cols].style.format("{:.3f}"))


## The held-out run

Pick your best configuration from the table above, then run it once on the held-out set.

Resist the urge to run all of them. Each pass leaks a little information about that set.

In [ ]:
# Choose the candidate label used by the following evaluation code.
# BEST is a plain string; setting it does not load or train any weights by itself.
# Read the next cell's actual branches: the 'quantised' label is listed here but that loader uses PyTorch base weights.
# This choice is meant to be made from dev evidence before examining final-set results.

BEST = "dpo"   # one of: stock_docs, sft, dpo, quantised
print("running held-out with:", BEST)


In [ ]:
# Load the selected candidate, define inference, then evaluate the final question set.
# transformers loads the causal LM/tokenizer; PEFT attaches a saved adapter; torch supplies tensors/MPS.
# tqdm.auto adds progress reporting. This cell loads model weights and, later in the cell, generates answers.
# The code is preserved as written: a missing chosen adapter currently leaves the base loaded.

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from tqdm.auto import tqdm
import torch

MODEL_PATH, DEVICE = CFG["model"]["base_path"], lab.device()
tok = AutoTokenizer.from_pretrained(MODEL_PATH)
if tok.pad_token is None: tok.pad_token = tok.eos_token
m = AutoModelForCausalLM.from_pretrained(MODEL_PATH, dtype=torch.float32).to(DEVICE)
# Only the 'sft' and 'dpo' branches attempt an adapter. Folder existence controls attachment;
# there is no separate quantized-model loader in this existing branch.
if BEST in ("sft","dpo") and (lab.ADAPTERS/BEST).exists():
    m = PeftModel.from_pretrained(m, str(lab.ADAPTERS/BEST))
# eval() switches modules such as dropout to inference behavior; it does not disable autograd.
# The no_grad context in ask() handles gradient recording during generation.
m.eval()

SYSTEM = ("You are the customer assistant for Meridian Motors. Answer only from "
          "Meridian's official documents. If you do not have the information, say so plainly.")
# Full source documents are included only for stock_docs. Adapter conditions omit them,
# so these settings do not isolate the effect of training under identical context.
if BEST == "stock_docs":
    SYSTEM += "\n\n=== DOCUMENTS ===\n" + "\n\n".join(p.read_text() for p in sorted(lab.RAW.glob("*.md")))


In [ ]:
# Define how the loaded model answers one question; the function body runs only when called.
# This uses the SYSTEM built above and preserves the existing truncation and decoding choices.


# The helper uses the selected global model, tokenizer, and SYSTEM prompt.
# It applies the chat template, creates token tensors, and keeps only the generated suffix.
def ask(q):
    t = tok.apply_chat_template([{"role":"system","content":SYSTEM},{"role":"user","content":q}],
                                tokenize=False, add_generation_prompt=True)
    # truncation=True cuts a long input to the tokenizer's configured side at 3,072 tokens.
    # This can remove context silently; the original behavior is documented here, not changed.
    i = tok(t, return_tensors="pt", truncation=True, max_length=3072).to(DEVICE)
    # The context manager avoids building a gradient graph. **i unpacks input_ids/attention_mask.
    # do_sample=False chooses greedy decoding; max_new_tokens caps only the new answer tokens.
    with torch.no_grad():
        o = m.generate(**i, max_new_tokens=CFG["inference"]["max_new_tokens"], do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(o[0][i["input_ids"].shape[1]:], skip_special_tokens=True).strip()


In [ ]:
# Run the final-set evaluation for the selected configuration and save its responses.
# No such evaluation is performed merely by reading the notebook; executing this block generates answers.
# The literal-value/refusal scorer is reused so its known limitations remain relevant.


# Read the final evaluation split only when this cell is actually executed.
# These answers should assess a frozen choice; repeatedly tuning against them leaks final-set information.
fidx, held = lab.fact_index(), lab.eval_set("heldout")
rows, raw = [], []
for q in tqdm(held, desc="HELD-OUT"):
    a = ask(q["question"]); rows.append(lab.score_answer(a, q, fidx))
    raw.append({"id":q["id"],"question":q["question"],"answer":a,"answerable":q["answerable"]})
held_s = lab.summarise(rows)
lab.save_run("13_heldout", {"config": BEST, "summary": held_s, "answers": raw})
print(json.dumps(held_s, indent=2))


## Against the bar you set blind

In [ ]:
# Compare final-set metrics with the previously declared quality bar and dev measurements.
# BAR comes from the saved notebook-01 run. Each dictionary value is a boolean threshold check.
# `>=` means at least the required accuracy/refusal rate; `<=` means no more than an allowed error rate.
# This reads prior results and prints comparisons; it does not rerun the model.

BAR = runs["00_bar"]["bar"]
checks = {
 "accuracy_answerable >= %.2f" % BAR["accuracy_answerable"]:
     held_s["accuracy_answerable"] >= BAR["accuracy_answerable"],
 "hallucination <= %.2f" % BAR["hallucination_rate_answerable"]:
     held_s["hallucination_rate_answerable"] <= BAR["hallucination_rate_answerable"],
 "appropriate refusal >= %.2f" % BAR["appropriate_refusal_rate"]:
     held_s["appropriate_refusal_rate"] >= BAR["appropriate_refusal_rate"],
 "over-refusal <= %.2f" % BAR["max_over_refusal"]:
     held_s["over_refusal_rate"] <= BAR["max_over_refusal"],
}
# items() gives label/value pairs; the conditional expression prints PASS or FAIL.
# The `%` formatting inside the dictionary labels and f-strings here both insert readable numbers.
for k,v in checks.items(): print(f"  {'PASS' if v else 'FAIL'}  {k}")
print("\nheld-out vs dev for the same config (a large gap means you overfitted to dev):")
# The lookup maps BEST to the matching saved dev run name. Missing runs raise an error.
# A dev-to-final gap is evidence to investigate; sampling variation and prompt differences can also contribute.
dev_key = {"dpo":"10_dpo_eval","sft":"08_sft_eval","stock_docs":"03_stock_with_docs","quantised":"11_quantised_eval"}[BEST]
for c in cols:
    print(f"  {c:32s} dev {runs[dev_key]['summary'][c]:.3f}  ->  heldout {held_s[c]:.3f}")


## Read the held-out failures, once

In [ ]:
# Read the actual final-set mistakes rather than relying only on aggregate percentages.
# zip(raw, rows) aligns each saved answer with the score produced for it in the same loop.
# Only incorrect scored answers are printed; the label separates answerable and unanswerable cases.
# The answer display is shortened to 150 characters; the saved raw answer is unchanged.

for r, sc in zip(raw, rows):
    if not sc["correct"]:
        print(f"[{'answerable' if r['answerable'] else 'UNANSWERABLE'}] {r['question']}")
        print(f"   -> {r['answer'][:150]}\n")


## Your recommendation

Write it yourself. This is the artefact worth having, and it is the thing an interviewer will probe.

A good recommendation names the decision, the evidence, the conditions under which it changes, and what you would do next. A recommendation **not** to deploy is a perfectly good outcome and is often the more senior answer.

In [ ]:
# Write a human decision record alongside the chosen configuration and final metrics.
# The empty strings are existing fields for your judgment; this code does not infer a deployment decision.
# Fill the evidence, risk, next experiment, and cost note before saving a meaningful recommendation.
# lab.save_run writes JSON; print(json.dumps(..., indent=2)) displays the same dictionary readably.

RECOMMENDATION = {
    "decision": "",            # "deploy" | "do not deploy" | "deploy with a human in the loop"
    "evidence": "",            # the two or three numbers that drove it
    "biggest_remaining_risk": "",
    "what_would_change_my_mind": "",
    "next_experiment": "",
    "cost_note": "",           # from notebook 07
}
lab.save_run("14_recommendation", {"recommendation": RECOMMENDATION, "config": BEST,
                                   "heldout_summary": held_s})
print(json.dumps(RECOMMENDATION, indent=2))


## The interview answers this lab produced

Check you can give each of these without notes:

1. **Why did you fine-tune at all?** Point to notebook 02. You measured the prompting ceiling first and it did or did not clear the bar.
2. **How do you know it worked?** A held-out number with a denominator, plus the dev-to-held-out gap.
3. **What is DPO and why not RLHF?** No reward model, no generation in the loop, closed-form optimal policy. Beta is the KL leash.
4. **How do you stop it hallucinating?** Two layers. A runtime validator that refuses uncited claims, and DPO that makes fabrication less likely in the first place.
5. **What does it cost?** Cost per accepted answer from notebook 07, with the assumptions stated.
6. **What would stop you shipping?** Your answer above.

The last one is the one most candidates cannot answer.